# pymefpp_swarp


This script is used after the pymefpp_scamp script. It does the resampling and combination of the files created by mefpp_scamp

The output are the coadd.fits and coadd.weight.fits files

In [ ]:
import os
import glob
from textwrap import wrap
from astropy.io import fits
import numpy as np
import subprocess
import builtins
import re
import copy
import types

# use logging for messages
import logging
logging.basicConfig(
#    filename="test.log",
    level=logging.INFO,
    format="%(asctime)s:%(levelname)s:%(message)s"
    )

cwd = os.getcwd()
logging.info('cwd: '.format(cwd))

In [ ]:
# ======================================================================
# A utility function
# ======================================================================
def runCmd( str_cmd, p_shell=True ):
    """ 
    A wrapper to run system commands.
      
    Parameters
    ----------
    str_cmd: str      
      Command string to be executed in the shell
    
    p_shell: bool
      If True (default), command will be executed through the shell, and all 
      cout/cerr messages will be available.
      If False, exception is the only way to find out problems during the call.

    Returns
    -------
      0 if some error, or 1 if all was OK

    Notes
    -----
    TODO:
      - allow to launch commands in background 
      - best checking of error when shell=True
    """
           
    logging.info("runCmd: Running command: {0}".format(str_cmd))
    
    try:
        p = subprocess.Popen(str_cmd, bufsize = 0, shell = p_shell, stdin = subprocess.PIPE, 
                             stdout = subprocess.PIPE, stderr = subprocess.PIPE, 
                             close_fds = True)
    except:
        logging.info("runCmd: Some error while running command...")
        raise
       
    #Warning
    #We use communicate() rather than .stdin.write, .stdout.read or .stderr.read 
    #to avoid deadlocks due to any of the other OS pipe buffers filling up and 
    #blocking the child process.(Python Ref.doc)

    (stdoutdata, stderrdata) = p.communicate()
    err = stdoutdata.decode('ascii') + " " + stderrdata.decode('ascii')

    # IMPORTANT: Next checking (only available when shell=True) not always detect all kind of errors !!
    if err.count('WARNING: Significant inaccuracy'):
        logging.info("runCmd: canno't get accuracy astrometric calibration")
        return 2
    elif (err.count('ERROR ') or err.count('error ') or err.count("Error ")\
      or err.count('*Error*')\
      or err.count('Segmentation fault') or err.count("command not found") \
      or err.count('No source found') \
      or err.count("No such file or directory")
      or err.count('WARNING: Not enough matched detections') # SCAMP
      or err.count('WARNING: Significant inaccuracy likely to occur in projection')): #SCAMP
        logging.info("An error happened while running command --> {0}".format(err))
        logging.info(stderrdata.decode('ascii'))
        return 1 # ERROR
    else:
        logging.info(stdoutdata.decode('ascii'))
        return 0 # NO ERROR
    


In [ ]:
# adapted to python3 from the PAPI pipeline

# ======================================================================

__version__ = "0.1 (2019-08-13)"



class SWARPException(Exception):
    pass

# ======================================================================

class SWARP(object):
    """
    A wrapper class to transparently use SWARP.

    """

    _SW_config = { 

#------------------------------Output-------------------------------------------
    
        "IMAGEOUT_NAME":
        {"comment": "Output filename",
         "value": "coadd.fits"},
        
        "WEIGHTOUT_NAME":
        {"comment": "Output weight-map filename",
         "value": "coadd.weight.fits"},
        
        "HEADER_ONLY":
        {"comment": "Only a header as an output file (Y/N)?",
         "value": "N"},
        
        "HEADER_SUFFIX":
        {"comment": 'Filename extension for additional headers',
         "value": ".head"},
         
#------------------------------- Input Weights --------------------------------        
        
        "WEIGHT_TYPE":
        {"comment": "BACKGROUND,MAP_RMS,MAP_VARIANCE or MAP_WEIGHT",
         "value": "NONE"},
        
        "WEIGHT_SUFFIX":
        {"comment": "Suffix to use for weight-maps",
         "value": ".weight.fits"},
        
        "WEIGHT_IMAGE":
        {"comment": "Weightmap filename if suffix not used (all or for each weight-map)",
         "value": ""},
        
        "WEIGHT_THRESH":
        {"comment": "Bad pixel weight-threshold",
         "value": ""},

#------------------------------- Co-addition ----------------------------------
        
        "COMBINE":
        {"comment": 'Combine resampled images (Y/N)?',
         "value": 'Y'},
        
        "COMBINE_TYPE":
        {"comment": "MEDIAN,AVERAGE,MIN,MAX,WEIGHTED,CHI2 or SUM",
         "value": "MEDIAN"},
        
        "BLANK_BADPIXELS":
        {"comment": "Set to 0 pixels having a weight of 0",
         "value": "N"},

#-------------------------------- Astrometry ----------------------------------

        "CELESTIAL_TYPE":
        {"comment": "NATIVE, PIXEL, EQUATORIAL,GALACTIC,ECLIPTIC, or SUPERGALACTIC",
         "value": "NATIVE"},

        "PROJECTION_TYPE":
        {"comment": "Any WCS projection code or NONE",
         "value": "TAN"},

        "PROJECTION_ERR":
        {"comment": "Maximum projection error (in output pixels), or 0 for no approximation",
         "value": 0.001 },

        "CENTER_TYPE":
        {"comment": "MANUAL, ALL or MOST",
         "value": "ALL"},

        "CENTER":
        {"comment": "Coordinates of the image center",
         "value": "00:00:00.0, +00:00:00.0"},

        "PIXELSCALE_TYPE":
        {"comment": "MANUAL,FIT,MIN,MAX or MEDIAN",
         "value": "MEDIAN"},
        
        "PIXEL_SCALE":
        {"comment": "Pixel scale",
         "value": 0},

        "IMAGE_SIZE":
        {"comment": "Image size (0 = AUTOMATIC)",
         "value": 0},

#-------------------------------- Resampling ----------------------------------
       
        "RESAMPLE":
        {"comment": "Resample input images (Y/N)?",
         "value": "Y"},

        "RESAMPLE_DIR":
        {"comment": "Directory path for resampled images",
         "value": "."},

        "RESAMPLE_SUFFIX":
        {"comment": "filename extension for resampled images",
         "value": ".resamp.fits"},

        "RESAMPLING_TYPE":
        {"comment": "NEAREST,BILINEAR,LANCZOS2,LANCZOS3or LANCZOS4 (1 per axis)",
         "value": "LANCZOS3"},

        "OVERSAMPLING":
        {"comment": "Oversampling in each dimension 0 = automatic)",
         "value": 0},

        "INTERPOLATE":
        {"comment": "Interpolate bad input pixels (Y/N)? (all or for each image)",
         "value": "Y"},

        "FSCALASTRO_TYPE":
        {"comment": "NONE,FIXED, or VARIABLE",
         "value": "FIXED"},

        "FSCALE_KEYWORD":
        {"comment": "FITS keyword for the multiplicative factor applied to each input image",
         "value": "FLXSCALE"},

        "FSCALE_DEFAULT":
        {"comment": "Default FSCALE value if not in header",
         "value": 1.0},

        "GAIN_KEYWORD":
        {"comment": "FITS keyword for effect. gain (e-/ADU)",
         "value": "GAIN"},

        "GAIN_DEFAULT":
        {"comment": "Default gain if no FITS keyword found 0 = infinity (all or for each image)",
         "value": 1.0},

        "SATLEV_KEYWORD":
        {"comment": "FITS keyword for saturation level (ADU)",
         "value": "SATURATE"},

        "SATLEV_DEFAULT":
        {"comment": "Default saturation if no FITS keyword",
         "value": 50000.0},

#--------------------------- Background subtraction ---------------------------

        "SUBTRACT_BACK":
        {"comment": "Subtraction sky background (Y/N)? (all or for each image)",
         "value": "Y"},

        "BACK_TYPE":
        {"comment": "AUTO or MANUAL (all or for each image)",
         "value": "AUTO"},

        "BACK_DEFAULT":
        {"comment": "Default background value in MANUAL (all or for each image)",
         "value": 0.0},

        "BACK_SIZE":
        {"comment": "Background mesh size (pixels) all or for each image)",
         "value": 128},
         
        "BACK_FILTERSIZE":
        {"comment": "Background map filter range (meshes) (all or for each image)",
         "value": 3},

        "BACK_FILTTHRESH":
        {"comment": "Threshold above which the background-map filter operates",
         "value": 0.0},
         
#------------------------------ Memory management -----------------------------

        "VMEM_DIR":
        {"comment": "Directory path for swap files",
         "value": "."},

        "VMEM_MAX":
        {"comment": "Maximum amount of virtual memory (MB)",
         "value": 32767},
         
        "MEM_MAX":
        {"comment": "Maximum amount of usable RAM (MB)",
         "value": 16384},

        "COMBINE_BUFSIZE":
        {"comment": "RAM dedicated to co-addition(MB)",
         "value": 8192},
        
#------------------------------ Miscellaneous ---------------------------------
        
        "DELETE_TMPFILES":
        {"comment": "Delete temporary resampled FITS files (Y/N)?",
         "value": "N"},
        
        "COPY_KEYWORDS":
        {"comment": 'List of FITS keywords to propagate from the input to the output headers',
         "value": "OBJECT,RA,DEC,OBJECT,PIXSCALE,ROT-RTA,INSTRUME,FILTER,TELESCOPE"},
        
        "WRITE_FILEINFO":
        {"comment": "Write information about each input file in the output image header?",
         "value": "Y"},
        
        "WRITE_XML":
        {"comment": 'Write XML file (Y/N)?',
         "value": "Y"},
        
        "XML_NAME":
        {"comment": "Filename for XML output",
         "value": "swarp.xml"},
        
        "XSL_URL":
        {"comment": "Filename for XSL style-sheet",
         "value": "file:///usr/share/swarp/swarp.xsl"},
        
        "VERBOSE_TYPE":
        {"comment": "QUIET,NORMAL or FULL",
         "value": "FULL"},
        
        "NNODES":
        {"comment": "Number of nodes (for clusters)",
         "value": 1},
        
        "NODE_INDEX":
        {"comment": "Node index (for clusters)",
         "value": 0},

        "NTHREADS":
        {"comment": "1 single thread",
         "value": 32},
        

        # -- Extra-keys (will not be saved in the main configuration file

        "CONFIG_FILE":
        {"comment": '[Extra key] name of the main configuration file',
         "value": "SWARP.conf"}
    }
        
    
    # -- Special config. keys that should not go into the config. file.

    _SW_config_special_keys = ["CONFIG_FILE"]



    def __init__(self):
        """
        SWARP class constructor.
        If a specific config_file is provided, it is used.
        """

        self.config = (
            dict([(k, copy.deepcopy(SWARP._SW_config[k]["value"]))\
                  for k in SWARP._SW_config.keys()]))

        # Extra config parameters that will be added/updated to the current 
        # values of the config file
        self.ext_config = {}       

        self.program = None
        self.version = None


    def setup(self, path=None):
        """
        Look for SWARP program ('swarp').
        If a full path is provided, only this path is checked.
        Raise a SWARPException if it failed.
        Return program and version if it succeed.
        """

        # -- Finding SWARP program and its version
        # first look for 'swarp'

        candidates = ['swarp']

        if (path):
            candidates = [path]
        
        selected = None
        for candidate in candidates:
            try:
                p = subprocess.Popen (candidate, shell = True, bufsize = 0,
                    stdin = subprocess.PIPE, stdout = subprocess.PIPE, 
                    stderr = subprocess.STDOUT, close_fds = True)
                
                versionline = p.communicate()[0]
                
                if (versionline.decode('ascii').find("SWarp") != -1):
                    selected=candidate
                    break
            except IOError:
                continue
                
        if not(selected):
            raise Exception("""
                  Cannot find SWARP program. Check your PATH,
                  or provide the SWARP program path in the constructor.
                  """)

        _program = selected

        #print versionline
        _version_match = re.search("[Vv]ersion ([0-9\.])+", versionline.decode('ascii'))
        if not _version_match:
            raise Exception("Cannot determine SWARP version.")

        _version = _version_match.group()[8:]
        if not _version:
            raise Exception("Cannot determine SWARP version.")

        # print "Use " + self.program + " [" + self.version + "]"

        return _program, _version



    def update_config(self):
        """
        Update the configuration files according to the current in-memory SWARP 
        configuration.
        """
        

        # -- Write main configuration file

        main_f = builtins.open(self.config['CONFIG_FILE'], 'w')

        for key in self.config.keys():
            if (key in SWARP._SW_config_special_keys):
                continue

            #if (key == "CENTER"): # tuple instead of a single value
            #    value = ",".join(map(str, self.config[key]))
            #else:
            value = str(self.config[key])
            
            print(("%-16s       %-16s # %s" % 
                          (key, value, SWARP._SW_config[key]['comment'])), file=main_f)

        main_f.close()


    def run(self, file_list, updateconfig=True, clean=False, path=None):
        """
        Description
        -----------
        
        Run SWARP for a given list of files (fits files), and it can be one 
        single file.

        Parameters
        ----------
        
        updateconfig: bool
            If True (default), the configuration files will be updated before 
            running SWARP.

        clean: bool
            If clean is True (default: False), configuration files (if any) will 
            be deleted after SWARP terminates.

        path: str
            Path to the 'swarp' application (binary file) in the system.
        """

        if updateconfig:
            self.update_config()

        # Try to find SWARP program
        # This will raise an exception if it failed
        
        self.program, self.version = self.setup(path)
        
        # check how many files in the input
        if type(file_list) == list:
            my_files=""
            for file in file_list:
                my_files = my_files + " " + file
        else:
            my_files = file_list # a single file


        # Compound extra config command line args
        ext_args=""
        for key in self.ext_config.keys():
            ext_args=ext_args + " -" + key+ " " + str(self.ext_config[key])

       
        commandline = (
            self.program + " -c " + self.config['CONFIG_FILE'] + " " + ext_args + " " + my_files)
        

        #print commandline

        #rcode = os.system(commandline)
        rcode = runCmd(commandline)
        
        if (rcode!=0):
            raise Exception("SWARP command [%s] failed." % str(commandline))
            
        if clean:
            self.clean()



    def clean(self, config=True):
        """
        Remove the generated SWARP files (if any).
        If config is True, remove generated configuration files.
        """

        try:
            if (config):
                os.unlink(self.config['CONFIG_FILE'])
                
        except OSError:
            pass



# ======================================================================

In [ ]:
# Here we do the combination of the frames using swarp

reduced_dir = '/home/fselman/local_storage/omegacam/reduced/science/CP_Pup_081146p060_m352105p0/NB_659/2017-03-13T01:57:49/'
current_dir = '/home/fselman/local_storage/omegacam/'
logging.info('current directory: {0}'.format(os.getcwd()))
os.chdir(reduced_dir)
logging.info('current directory: {0}'.format(os.getcwd()))
list_to_swarp = glob.glob('OMEGA.*_pre')

frame_list = 'frames_to_swarp.list'
with open(frame_list, 'w') as f:
    files_to_swarp = []
    for afile in list_to_swarp:
        file_name = afile+'/'+afile+'_wcs.fits'
        files_to_swarp.append(file_name)
        print(file_name, file=f)

swarp = SWARP()
# Using a specific config file (updateconfig=False)
swarp.config['CONFIG_FILE']="./swarp.conf"
try:
    os.makedirs('./resamp')
except OSError:
    pass
swarp.config['RESAMPLE_DIR'] = './resamp'
swapdir = './swap'
try:
    os.makedirs(swapdir)
except OSError:
    pass
swarp.config['VMEM_DIR'] = './swap'

# we set some reasonable parameters
swarp.config['BACK_SIZE'] = 128           # size of blocks of pixels for bckg estimation
swarp.config['BACK_FILTERSIZE'] = 5       # number of mesh points to interpolate sky background
swarp.config['COMBINE_TYPE'] = 'MEDIAN'   # AVERAGE, CHI2, MEDIAN, MIN, MAX, SUM, WEIGHTED

swarp.run('@'+ frame_list, updateconfig=True, clean=False)